# E-commerce product recommendation benchmark (Kazakhstan)

This notebook uses the **real transaction CSV** `dataset/kz.csv.gz` (lossless gzip of the original CSV), downloaded from the public Kaggle dataset [eCommerce purchase history from electronics store](https://www.kaggle.com/datasets/mkechinov/ecommerce-purchase-history-from-electronics-store). It does not generate synthetic rows.

The notebook inspects the data, documents the time split, optionally reruns all 11 regression models with 10-fold user-grouped validation, and displays the results saved in `results/model_results.json`. The companion implementation is `recommendation_benchmark.py`.

**Important target limitation:** the source contains purchases but no impression logs or explicit non-purchases. Unobserved user-item pairs sampled as label 0 are assumed negatives, not confirmed dislikes. The source does not establish lecture-cohort uniqueness; ask the instructor to confirm it.

## Reproduce

Keep this notebook beside `dataset/`, `recommendation_benchmark.py`, and `results/`. Install dependencies from `requirements.txt`. The full benchmark takes longer than a quick notebook preview because it fits 11 models across 10 folds. Set `RUN_FULL_BENCHMARK = True` in the benchmark cell to rerun it.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

# Find the project root even when Jupyter starts in a parent directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if ((p / 'dataset' / 'kz.csv').exists() or (p / 'dataset' / 'kz.csv.gz').exists())), None)
if ROOT is None:
    raise FileNotFoundError('Cannot find dataset/kz.csv or dataset/kz.csv.gz. Open this notebook from the project folder.')
DATA_PATH = ROOT / 'dataset' / 'kz.csv'
if not DATA_PATH.exists():
    DATA_PATH = ROOT / 'dataset' / 'kz.csv.gz'
SCRIPT_PATH = ROOT / 'recommendation_benchmark.py'
RESULT_PATH = ROOT / 'results' / 'model_results.json'
print('Project:', ROOT)
print('Dataset:', DATA_PATH, f'({DATA_PATH.stat().st_size / (1024**2):.1f} MiB)')
print('Source: https://www.kaggle.com/datasets/mkechinov/ecommerce-purchase-history-from-electronics-store')

## Inspect the original CSV

The preview and schema below are read from the supplied Kaggle file itself.

In [ ]:
preview = pd.read_csv(DATA_PATH, nrows=5)
print('Columns:', preview.columns.tolist())
display(preview)

# Use the benchmark's production cleaning/splitting function on the complete CSV.
import recommendation_benchmark as benchmark
history, future, data_summary = benchmark.load_and_split(DATA_PATH, '2020-09-01')
data_profile = pd.Series(data_summary, name='value').to_frame()
display(data_profile)
del history, future  # Free memory before optional model refitting.

## Model training and validation

The training script creates user/item features only from history before 2020-09-01. It samples up to four unseen candidate items per positive pair and uses `GroupKFold(n_splits=10)` grouped by user. The target is binary (observed later purchase vs. assumed negative), scored as a regression problem per the assignment.

In [ ]:
RUN_FULL_BENCHMARK = False  # Change to True to refit all 11 models on the real CSV.
if RUN_FULL_BENCHMARK:
    completed = subprocess.run(
        [sys.executable, str(SCRIPT_PATH), '--data', str(DATA_PATH),
         '--max-positive-pairs', '6000', '--negatives', '4', '--seed', '42',
         '--output', str(RESULT_PATH)],
        cwd=ROOT, text=True, capture_output=True, check=True
    )
    print('Benchmark completed:')
    for line in completed.stdout.splitlines():
        if line.startswith('COMPLETED '):
            print(line)
else:
    print('Using the completed benchmark output in', RESULT_PATH)
    print('Set RUN_FULL_BENCHMARK = True to refit every model.')

## Results

The table is loaded from the machine-readable output created by the benchmark script. RMSE and R² are the assignment regression metrics; NDCG@10 and HitRate@10 add top-N recommendation context.

In [ ]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
results = pd.DataFrame(result['results'])
display(results[['algorithm', 'features', 'targets', 'folds',
                 'rmse_mean', 'rmse_std', 'r2_mean', 'r2_std',
                 'ndcg_at_10_mean', 'hit_rate_at_10_mean']].round(4))

print('Target:', result['evaluation']['target_definition'])
print('Validation:', result['evaluation']['split'])
print('Sampled positives:', result['evaluation']['positive_user_item_pairs'])
print('Candidate rows:', result['evaluation']['sampled_candidates'])

## Interpretation and limitations

In this fixed configuration, LightGBM had the lowest mean RMSE (0.3349) and highest mean R² (0.2444); XGBoost was effectively tied on RMSE (0.3350). Gradient Boosting Regression had the highest NDCG@10 (0.7467). These results show that prediction error and top-10 ordering are different objectives.

The benchmark is offline and conditional on sampled candidates. Missing purchases are not verified rejections; most source events have no user ID; the log is from 2020; no tuning, significance test, external validation, or online experiment was done. Confirm dataset uniqueness and licensing before submission, and independently review citations and the course AI policy.